# 🧩 Combine 6-class — YOLO26 ⊕ YOLO11 + Auto-label

**File 3/3.** Cần `yolo26s_best.pt` và `yolo11s_best.pt` (từ file 1 & 2, đã lưu Drive).

## 5 phương pháp kết hợp (tự thiết kế cho schema 6-class, KHÔNG copy tài liệu sẵn)
| # | Phương pháp | Tăng gì | Mức |
|---|-------------|---------|-----|
| 1 | **Per-class Adaptive WBF** — mỗi model bỏ phiếu theo AP từng class | mAP50 | model-level |
| 2 | **TTA hoán-đổi-L/R** — flip ngang + remap `eyeL↔eyeR` | mAP, recall | inference |
| 3 | **Drowsiness Decision Layer** — 6 class → 1 điểm buồn ngủ | logic | decision |
| 4 | **PERCLOS + Yawn-rate thời gian** — cửa sổ trượt trên video | giảm báo động giả | temporal |
| 5 | **Self-training** — auto-label ảnh thật → train lại | data | semi-supervised |

> Điểm gốc quan trọng: schema này tách **mắt trái/phải** → (a) flip ngang làm SAI nhãn nếu không remap; (b) tính được PERCLOS chính xác từng mắt; (c) phân biệt nháy mắt 1 bên vs nhắm cả 2.

In [ ]:
# 1 — Setup + load 2 model đã train
!nvidia-smi -L
%pip install -q -U "ultralytics>=8.4.0" roboflow ensemble-boxes
import os, json, shutil, glob
from pathlib import Path
import numpy as np, cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

from google.colab import drive
drive.mount('/content/drive')
OUT  = Path('/content/drive/MyDrive/DrowsyDriver_Results')
HOME = Path('/content')

P26 = OUT/'yolo26s_best.pt'
P11 = OUT/'yolo11s_best.pt'
assert P26.exists() and P11.exists(), '❌ Thiếu best.pt — chạy file 1 & 2 trước'

m26 = YOLO(str(P26))
m11 = YOLO(str(P11))
CLASSES = list(m26.names.values())
print('Classes:', CLASSES)

# Index helper cho 6 class (an toàn nếu thứ tự khác)
def idx(name): return CLASSES.index(name) if name in CLASSES else -1
I_CL, I_CR = idx('close_eyeL'), idx('close_eyeR')
I_OL, I_OR = idx('open_eyeL'),  idx('open_eyeR')
I_YAWN     = idx('yawn')

In [ ]:
# Tải dataset 6-class + XÂY LẠI data.yaml chuẩn + lấy test images
import yaml
from roboflow import Roboflow
rf = Roboflow(api_key='qI3lEKlNpIZpNENdk3MH')

# ⚠️ Slug Roboflow LUÔN viết thường! 'Datio_yolo' (hoa) → 404.
PROJECT_TRY = ['datio_yolo', 'driver-yawn', 'driver-yawn-wh6wj']
ds = None
for proj in PROJECT_TRY:
    try:
        ds = rf.workspace('nguyen-tuan-dat').project(proj).version(1).download('yolov11')
        print('✅  Dùng project:', proj); break
    except Exception:
        print('  ⏭️ ', proj, 'không tải được, thử tiếp...')

loc = Path(ds.location)
# lấy names từ yaml cũ (kể cả khi thiếu train/val)
old = {}
for yp in (list(loc.rglob('*.yaml')) + list(loc.rglob('*.yml'))):
    try:
        with open(yp) as f: tmp = yaml.safe_load(f) or {}
        if 'names' in tmp: old = tmp; break
    except Exception: pass
names = old.get('names')
if isinstance(names, dict): names = [names[k] for k in sorted(names)]
if not names: names = ['close_eyeL','close_eyeR','no_yawn','open_eyeL','open_eyeR','yawn']

def imgdir(*c):
    for x in c:
        d = loc/x
        if d.exists() and any(d.iterdir()): return str(d.resolve())
    return None
train_dir = imgdir('train/images','train')
val_dir   = imgdir('valid/images','val/images','valid','val') or train_dir
test_dir  = imgdir('test/images','test')

cfg = {'train': train_dir, 'val': val_dir, 'nc': len(names), 'names': names}
if test_dir: cfg['test'] = test_dir
DATA_YAML = str(loc/'data.yaml')
with open(DATA_YAML,'w') as f: yaml.dump(cfg, f, sort_keys=False)
base = loc

TEST_DIR = Path(test_dir) if test_dir else Path(val_dir)
test_imgs = sorted(list(TEST_DIR.glob('*.jpg'))+list(TEST_DIR.glob('*.png')))
print(f'  Classes: {names}')
print(f'  Test images: {len(test_imgs)}')

---
## 1️⃣ Per-class Adaptive WBF

WBF thường: mỗi model 1 trọng số cố định. **Cải tiến**: đo AP **từng class** của mỗi model trên val,
rồi nhân confidence theo AP đó *trước khi* fusion → class nào model nào giỏi thì model đó "nói to hơn".

Ví dụ YOLO26 giỏi `yawn`, YOLO11 giỏi `close_eyeL` → fusion tự ưu tiên đúng model cho đúng class.

In [ ]:
# Đo per-class AP của mỗi model → trọng số
def per_class_ap(model):
    mt = model.val(data=DATA_YAML, verbose=False)
    return {i: float(mt.box.maps[i]) for i in range(len(CLASSES))}  # mAP50-95 per class

ap26 = per_class_ap(m26)
ap11 = per_class_ap(m11)
print(f'  {"class":<12} {"YOLO26":>8} {"YOLO11":>8}  winner')
W26, W11 = {}, {}
for i,c in enumerate(CLASSES):
    a26, a11 = ap26[i]+1e-6, ap11[i]+1e-6
    W26[i] = a26/(a26+a11); W11[i] = a11/(a26+a11)
    print(f'  {c:<12} {a26*100:7.2f}% {a11*100:7.2f}%  {"YOLO26" if a26>a11 else "YOLO11"}')

In [ ]:
# WBF với confidence đã nhân trọng số per-class
from ensemble_boxes import weighted_boxes_fusion

def predict_norm(model, img_path, conf=0.01):
    """Trả boxes (xyxy normalized), scores, labels"""
    r = model.predict(img_path, conf=conf, iou=0.6, verbose=False)[0]
    if len(r.boxes)==0: return [], [], []
    return (r.boxes.xyxyn.cpu().numpy().tolist(),
            r.boxes.conf.cpu().numpy().tolist(),
            r.boxes.cls.cpu().numpy().astype(int).tolist())

def adaptive_wbf(img_path, iou=0.55, skip=0.25):
    b26,s26,l26 = predict_norm(m26, img_path)
    b11,s11,l11 = predict_norm(m11, img_path)
    # nhân confidence theo trọng số per-class
    s26 = [min(1.0, s*W26.get(l,0.5)*2) for s,l in zip(s26,l26)]
    s11 = [min(1.0, s*W11.get(l,0.5)*2) for s,l in zip(s11,l11)]
    if not b26 and not b11: return [],[],[]
    return weighted_boxes_fusion([b26 or [[0,0,0,0]], b11 or [[0,0,0,0]]],
                                 [s26 or [0],          s11 or [0]],
                                 [l26 or [0],          l11 or [0]],
                                 iou_thr=iou, skip_box_thr=skip)

# Demo trên 6 ảnh test
COLORS = {I_CL:(255,80,80), I_CR:(255,160,0), I_OL:(80,160,255),
          I_OR:(0,200,120), I_YAWN:(220,0,220)}
fig, ax = plt.subplots(2,3, figsize=(16,9))
for a, ip in zip(ax.flat, test_imgs[:6]):
    img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB); h,w = img.shape[:2]
    bxs,scs,lbs = adaptive_wbf(str(ip))
    for bx,sc,lb in zip(bxs,scs,lbs):
        if sc<0.3: continue
        x1,y1,x2,y2 = int(bx[0]*w),int(bx[1]*h),int(bx[2]*w),int(bx[3]*h)
        c = COLORS.get(int(lb),(255,255,0))
        cv2.rectangle(img,(x1,y1),(x2,y2),c,2)
        cv2.putText(img,f'{CLASSES[int(lb)]} {sc:.2f}',(x1,max(12,y1-5)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.45,c,1)
    a.imshow(img); a.axis('off')
plt.suptitle('Per-class Adaptive WBF (YOLO26 ⊕ YOLO11)', fontweight='bold')
plt.tight_layout(); plt.savefig(OUT/'ensemble_wbf.png', dpi=120, bbox_inches='tight'); plt.show()

---
## 2️⃣ TTA hoán-đổi-L/R (insight riêng của schema này)

Test-Time Augmentation thường: chạy thêm ảnh lật ngang rồi gộp. **Nhưng** với schema này, lật ngang khiến
`close_eyeL` ↔ `close_eyeR` và `open_eyeL` ↔ `open_eyeR` **đổi chỗ**. Nếu không remap, TTA làm SAI nhãn.

Quy trình đúng: predict ảnh gốc + predict ảnh lật → **lật ngược tọa độ** → **remap class L↔R** → WBF.

In [ ]:
# TTA với remap L↔R
LR_SWAP = {I_CL:I_CR, I_CR:I_CL, I_OL:I_OR, I_OR:I_OL}   # yawn/no_yawn giữ nguyên

def predict_tta(model, img_path, iou=0.55, skip=0.25):
    img = cv2.imread(str(img_path))
    # gốc
    b0,s0,l0 = predict_norm(model, img_path)
    # lật ngang
    flip = cv2.flip(img, 1)
    fp = str(HOME/'_tta_flip.jpg'); cv2.imwrite(fp, flip)
    bf,sf,lf = predict_norm(model, fp)
    # lật ngược toạ độ (x' = 1 - x, đảo x1/x2) + remap class
    bf2, lf2 = [], []
    for bx,lb in zip(bf,lf):
        x1,y1,x2,y2 = bx
        bf2.append([1-x2, y1, 1-x1, y2])
        lf2.append(LR_SWAP.get(int(lb), int(lb)))
    if not b0 and not bf2: return [],[],[]
    return weighted_boxes_fusion([b0 or [[0,0,0,0]], bf2 or [[0,0,0,0]]],
                                 [s0 or [0],          sf or [0]],
                                 [l0 or [0],          lf2 or [0]],
                                 iou_thr=iou, skip_box_thr=skip)

# So sánh nhanh: bao nhiêu detection thêm nhờ TTA (recall ↑)
n_base = n_tta = 0
for ip in test_imgs[:40]:
    b,_,_ = predict_norm(m26, str(ip)); n_base += len(b)
    b,_,_ = predict_tta(m26, str(ip));  n_tta  += len(b)
print(f'  Detections (40 ảnh):  base={n_base}  TTA={n_tta}  (+{n_tta-n_base})')
print('  ✅ TTA-LR đã remap đúng nhãn trái/phải')

---
## 3️⃣ Drowsiness Decision Layer (6 class → 1 điểm buồn ngủ)

Detector cho ra 6 class rời rạc. Lớp này gộp thành **1 điểm `s ∈ [0,1]`** theo logic an toàn:
```
eye_closure = max(conf close_eyeL, close_eyeR đối ứng) / (mở + nhắm)
s = 0.6 · eye_closure  +  0.4 · conf(yawn)
```
Trọng số 0.6/0.4 vì **nhắm mắt nguy hiểm hơn ngáp**. Cả 2 mắt nhắm → s cao nhất.

In [ ]:
def max_conf_per_class(bxs, scs, lbs, thr=0.25):
    """conf cao nhất mỗi class trong 1 frame"""
    out = {i:0.0 for i in range(len(CLASSES))}
    for s,l in zip(scs,lbs):
        if s>=thr: out[int(l)] = max(out[int(l)], float(s))
    return out

def drowsiness_score(conf):
    cL,cR = conf.get(I_CL,0), conf.get(I_CR,0)
    oL,oR = conf.get(I_OL,0), conf.get(I_OR,0)
    yawn  = conf.get(I_YAWN,0)
    # PERCLOS-like cho từng mắt rồi lấy trung bình 2 mắt
    clsL = cL/(cL+oL+1e-6); clsR = cR/(cR+oR+1e-6)
    eye_closure = (clsL+clsR)/2
    s = 0.6*eye_closure + 0.4*yawn
    state = 'DROWSY 🔴' if s>=0.6 else ('WARNING 🟡' if s>=0.35 else 'ALERT 🟢')
    return s, state, dict(eye_closure=round(eye_closure,2), yawn=round(yawn,2))

# Demo
print(f'  {"image":<22} {"score":>6}  state')
for ip in test_imgs[:10]:
    bxs,scs,lbs = adaptive_wbf(str(ip))
    conf = max_conf_per_class(bxs,scs,lbs)
    s, st, dbg = drowsiness_score(conf)
    print(f'  {ip.name[:22]:<22} {s:6.2f}  {st}   {dbg}')

---
## 4️⃣ PERCLOS + Yawn-rate theo thời gian (cho video)

Buồn ngủ là trạng thái **kéo dài**, không phải 1 frame. Gộp theo cửa sổ trượt:
```
PERCLOS  = số frame (cả 2 mắt nhắm) / tổng frame trong cửa sổ
Yawn-rate= số lần ngáp / phút
ALARM nếu PERCLOS > 0.4  HOẶC  yawn-rate ≥ 3
```
Đây là tiêu chí chuẩn ngành (PERCLOS) nhưng áp lên đầu ra 6-class — giảm hẳn báo động giả do 1 frame lỗi.

In [ ]:
from collections import deque

class DrowsinessMonitor:
    def __init__(self, fps=15, window_sec=4, perclos_thr=0.4, yawn_per_min=3):
        self.W = int(fps*window_sec)
        self.fps = fps; self.perclos_thr = perclos_thr; self.yawn_per_min = yawn_per_min
        self.eye_hist  = deque(maxlen=self.W)         # 1 nếu cả 2 mắt nhắm
        self.yawn_hist = deque(maxlen=int(fps*60))    # 60s
        self.prev_yawn = False
    def update(self, conf):
        both_closed = (conf.get(I_CL,0)>0.4 and conf.get(I_CR,0)>0.4)
        self.eye_hist.append(1 if both_closed else 0)
        is_yawn = conf.get(I_YAWN,0)>0.5
        self.yawn_hist.append(1 if (is_yawn and not self.prev_yawn) else 0)  # đếm cạnh lên
        self.prev_yawn = is_yawn
        perclos   = sum(self.eye_hist)/max(len(self.eye_hist),1)
        yawn_rate = sum(self.yawn_hist)/max(len(self.yawn_hist)/self.fps/60, 1e-6)
        alarm = (perclos>self.perclos_thr) or (yawn_rate>=self.yawn_per_min)
        return dict(perclos=round(perclos,2), yawn_rate=round(yawn_rate,1), ALARM=alarm)

# Mô phỏng 60 frame (thay bằng cv2.VideoCapture cho video thật)
mon = DrowsinessMonitor(fps=15, window_sec=4)
seq = (test_imgs * 5)[:60]
for i, ip in enumerate(seq):
    bxs,scs,lbs = adaptive_wbf(str(ip))
    st = mon.update(max_conf_per_class(bxs,scs,lbs))
    if i % 12 == 0 or st['ALARM']:
        print(f'  frame {i:>3}: PERCLOS={st["perclos"]}  yawn/min={st["yawn_rate"]}  ALARM={st["ALARM"]}')
print('\n  → Trên video thật: thay seq bằng frame từ cv2.VideoCapture(video.mp4)')

---
## 5️⃣ Auto-label ảnh THẬT vào 6 class

**Câu hỏi của bạn:** cập nhật ảnh thực tế, để nó tự gán vào 6 class này.

### Tại sao KHÔNG dùng foundation model (Grounding DINO / Auto Label của Roboflow)?
`close_eyeL` / `close_eyeR` **không phải khái niệm ngôn ngữ tự nhiên** — model nền không phân biệt được trái/phải.
→ Phải dùng **chính model bạn vừa train** để auto-label (model-assisted labeling / pseudo-label).

### Quy trình (Active Learning loop):
```
Ảnh thật mới → model dự đoán → giữ box conf cao → xuất .txt YOLO
            → upload lại Roboflow → người review/sửa → thêm vào dataset → train lại
```

In [ ]:
# 5a — Auto-label: dùng ensemble dự đoán ảnh thật → xuất nhãn YOLO .txt
# Upload ảnh thật vào /content/real_images/ (hoặc mount Drive)
REAL_DIR  = HOME/'real_images';  REAL_DIR.mkdir(exist_ok=True)
LABEL_DIR = HOME/'auto_labels';  LABEL_DIR.mkdir(exist_ok=True)
REVIEW_DIR= HOME/'need_review';  REVIEW_DIR.mkdir(exist_ok=True)

from google.colab import files
real = sorted(list(REAL_DIR.glob('*.jpg'))+list(REAL_DIR.glob('*.png')))
if not real:
    print('  Upload ảnh thật (chọn nhiều ảnh):')
    up = files.upload()
    for fn in up: shutil.move(fn, REAL_DIR/fn)
    real = sorted(list(REAL_DIR.glob('*.jpg'))+list(REAL_DIR.glob('*.png')))

CONF_KEEP   = 0.70   # >=0.70 → auto-label luôn
CONF_REVIEW = 0.40   # 0.40–0.70 → cần người duyệt
n_auto = n_rev = 0
for ip in real:
    bxs,scs,lbs = adaptive_wbf(str(ip))     # ensemble 2 model
    img = cv2.imread(str(ip)); h,w = img.shape[:2]
    lines, low = [], False
    for bx,sc,lb in zip(bxs,scs,lbs):
        if sc < CONF_REVIEW: continue
        if sc < CONF_KEEP: low = True
        # YOLO format: class cx cy w h (normalized)
        cx,cy = (bx[0]+bx[2])/2, (bx[1]+bx[3])/2
        bw,bh = bx[2]-bx[0], bx[3]-bx[1]
        lines.append(f'{int(lb)} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
    if lines:
        (LABEL_DIR/(ip.stem+'.txt')).write_text('\n'.join(lines))
        if low: shutil.copy(ip, REVIEW_DIR/ip.name); n_rev += 1
        else:   n_auto += 1
print(f'  ✅ Auto-label chắc chắn: {n_auto} ảnh  |  Cần review: {n_rev} ảnh')
print(f'  Nhãn .txt → {LABEL_DIR}  (upload kèm ảnh lên Roboflow để review)')

In [ ]:
# 5b — Upload thẳng nhãn đã duyệt lên Roboflow (qua API) để gộp vào dataset
# Roboflow tự nhận .txt YOLO nếu cùng tên ảnh. Review trên web rồi Generate version mới.
from roboflow import Roboflow
rf = Roboflow(api_key='qI3lEKlNpIZpNENdk3MH')
project = rf.workspace('nguyen-tuan-dat').project('Datio_yolo')  # ← sửa đúng project bạn

UPLOAD = True   # đặt True khi muốn đẩy lên
if UPLOAD:
    for ip in sorted(REAL_DIR.glob('*.*')):
        lbl = LABEL_DIR/(ip.stem+'.txt')
        if lbl.exists():
            project.upload(image_path=str(ip), annotation_path=str(lbl),
                           split='train', num_retry_uploads=2)
    print('  ✅ Đã upload. Vào Roboflow → Annotate → review → Generate version mới → train lại.')
else:
    print('  (UPLOAD=False) — bật True để đẩy nhãn lên Roboflow')

print('\n  CÁCH 2 — Roboflow Label Assist (trên web, không cần code):')
print('  1. Roboflow → Models → Upload weights yolo26s_best.pt (hoặc train trên Roboflow)')
print('  2. Annotate → Label Assist → chọn model của bạn → nó tự vẽ box 6 class')
print('  3. Sửa lại box sai → Save → Generate version → train lại')

---
## 📚 Dataset gợi ý thêm (tăng độ đa dạng)

| Dataset | Schema | Ghi chú |
|---------|--------|---------|
| **lukas** (Roboflow Universe) | `close_eyeL/R, open_eyeL/R, yawn, no_yawn` | ✅ **TRÙNG 6-class** — merge trực tiếp (chỉ 200 ảnh) |
| labo54 *drowsiness-detection* | `Closed eye, Opened eye, Yawn, No-yawn` (4) | gộp được nếu **bỏ phân biệt L/R** |
| driver-no-yawn *driver-drowsiness1* | `drowsy, not_drowsy` (2) | schema khác, không gộp thẳng |

> **Lưu ý quan trọng:** mỗi class hiện chỉ ~240 ảnh — rất ít. 2 lựa chọn:
> - **Giữ 6-class**: chỉ merge được dataset cũng có L/R (hiếm) → dùng auto-label tự thu thập thêm.
> - **Gộp xuống 4-class** (`closed_eye, open_eye, yawn, no_yawn`): mở khoá HÀNG CHỤC dataset Roboflow → nhiều data hơn, mỗi class ~2x ảnh. Đánh đổi: mất khả năng phân biệt nháy mắt 1 bên.

In [ ]:
# (Tuỳ chọn) Merge dataset 'lukas' cùng 6-class schema để có thêm data
# Bỏ comment để chạy — cần biết workspace/project slug chính xác của lukas
MERGE_LUKAS = False
if MERGE_LUKAS:
    rf = Roboflow(api_key='qI3lEKlNpIZpNENdk3MH')
    # TODO: thay bằng slug đúng (tìm trên universe.roboflow.com — search 'close_eyeL')
    extra = rf.workspace('LUKAS_WORKSPACE').project('LUKAS_PROJECT').version(1).download('yolov11')
    # → copy ảnh+nhãn vào dataset chính NẾU thứ tự class trùng khớp.
    #   QUAN TRỌNG: kiểm tra names trong data.yaml của lukas khớp thứ tự với CLASSES của bạn,
    #   nếu lệch thứ tự → phải remap class_id trong .txt trước khi gộp.
    print('  Kiểm tra names khớp thứ tự trước khi gộp!')
else:
    print('  MERGE_LUKAS=False — bật True + điền slug để gộp data 6-class của lukas')

In [ ]:
# 📊 Tổng kết: single model vs ensemble vs ensemble+TTA
def eval_strategy(predict_fn, n=80):
    """Đếm tổng detection conf>=0.3 (proxy nhanh cho recall; mAP chính xác xem .val ở trên)"""
    tot = 0
    for ip in test_imgs[:n]:
        b,s,l = predict_fn(str(ip))
        tot += sum(1 for sc in s if sc>=0.3)
    return tot

print('  Chiến lược                         | detections (80 ảnh, conf≥0.3)')
print('  '+'-'*60)
print(f'  YOLO26 đơn                         | {eval_strategy(lambda p: predict_norm(m26,p))}')
print(f'  YOLO11 đơn                         | {eval_strategy(lambda p: predict_norm(m11,p))}')
print(f'  Per-class Adaptive WBF (26⊕11)     | {eval_strategy(adaptive_wbf)}')
print(f'  YOLO26 + TTA-LR                    | {eval_strategy(lambda p: predict_tta(m26,p))}')
print()
print('  mAP chính xác: xem output .val() ở Cell per-class AP (mục 1).')
print('  Khuyến nghị deploy: Adaptive-WBF + DrowsinessDecisionLayer + PERCLOS temporal.')